# 01 — Explore Hackathon Data

Explores the Challenge 1 dataset in S3 and validates the schema built in `backend/data/schema.py`.

**Run order:** Execute cells top-to-bottom. Requires valid AWS credentials in environment.

In [ ]:
import sys, os
sys.path.insert(0, '..')  # make backend importable from notebooks/

# Load .env if present
try:
    from dotenv import load_dotenv; load_dotenv('../.env')
except ImportError:
    pass

import json
from collections import defaultdict, Counter
from backend.config.aws_config import CHALLENGE_PREFIX, _get_bucket_name
from backend.data.s3_loader import list_challenge_files, download_file
from backend.data.data_loader import (
    load_club_season_stats, load_player_roster, load_player_stats,
    load_user_profiles, load_match, CLUB_ID_TO_NAME
)
print('Imports OK — bucket:', _get_bucket_name())

## 1. Full file tree grouped by subfolder

In [ ]:
all_keys = list_challenge_files()
print(f'Total objects in challenge: {len(all_keys)}\n')

groups = defaultdict(list)
for k in all_keys:
    rel = k.replace(CHALLENGE_PREFIX + '/', '')
    parts = rel.split('/')
    sub = '/'.join(parts[:2]) if len(parts) >= 2 else parts[0]
    groups[sub].append(rel)

for g in sorted(groups.keys()):
    items = groups[g]
    exts = set(r.rsplit('.', 1)[-1] if '.' in r else 'no-ext' for r in items)
    print(f'{g}: {len(items)} files  [{", ".join(sorted(exts))}]')
    for r in items[:3]:
        print(f'  {r}')
    if len(items) > 3:
        print(f'  ... and {len(items)-3} more')

## 2. File type summary

In [ ]:
ext_counts = Counter(k.rsplit('.', 1)[-1].lower() if '.' in k else 'no-ext' for k in all_keys)
print('File counts by type:')
for ext, cnt in sorted(ext_counts.items(), key=lambda x: -x[1]):
    print(f'  .{ext}: {cnt}')

club_ids = set()
for k in all_keys:
    if '/players/' in k and k.endswith('.xml'):
        fname = k.split('/')[-1]
        cid = fname.replace('01.05.', '').split('_')[0]
        if cid.startswith('DFL-CLU'):
            club_ids.add(cid)
print(f'\nUnique clubs in player feeds: {len(club_ids)}')
for cid in sorted(club_ids):
    print(f'  {cid}  →  {CLUB_ID_TO_NAME.get(cid, "unknown")}')

## 3. Bayern season stats — top 10 scorers

In [ ]:
stats = load_club_season_stats()
print(f'Bayern players with season stats: {len(stats)}\n')

top_scorers = sorted(stats.values(), key=lambda p: p.goals, reverse=True)[:10]
print(f'{"Player":<25} {"Goals":>6} {"Assists":>8} {"xG":>6} {"Apps":>5} {"Mins":>6}')
print('-' * 60)
for p in top_scorers:
    print(f'{p.name:<25} {p.goals:>6} {p.assists:>8} {p.xg:>6.2f} {p.appearances:>5} {p.normalized_minutes:>6}')

## 4. Player roster — sample club

In [ ]:
roster = load_player_roster('DFL-CLU-00000G')  # Bayern
print(f'Bayern roster: {len(roster)} players\n')
print(f'{"Name":<30} {"Position":<12} {"Nationality":<15} {"Height":>7}')
print('-' * 68)
for p in roster[:15]:
    print(f'{p.name:<30} {p.position:<12} {p.nationality:<15} {p.height_cm:>7}')

## 5. Sample match record

In [ ]:
match = load_match('DFL-MAT-J04034')  # Matchday 1: M'gladbach vs Leverkusen
print(f'Match: {match.home_team_name} vs {match.away_team_name}')
print(f'Result: {match.result}  |  Matchday: {match.matchday}')
print(f'Kickoff: {match.actual_kickoff}')
print(f'Stadium: {match.stadium_name} ({match.spectators:,} spectators)')
print(f'Formations: {match.home_formation} vs {match.away_formation}')
print(f'Home starters ({len(match.home_starters)}): {match.home_starters[:5]}...')
print(f'Drama score: {match.drama_score} (computed by scoring engine in Section 4)')

## 6. User profiles — sample

In [ ]:
profiles = load_user_profiles()
print(f'Total users: {len(profiles)}')

# Show a sample user
sample = next(iter(profiles.values()))
print(f'\nSample user: {sample.user_id[:16]}...')
print(f'  Club: {sample.favorite_club} ({sample.favorite_club_id})')
print(f'  Country: {sample.country}  |  Platform: {sample.platform}')
print(f'  Active months: {sample.active_months}')
print(f'  Total match-centre views: {sample.total_match_center_views}')
print(f'  Stats focus: {sample.stats_focus_ratio:.1%}  Ticker: {sample.ticker_focus_ratio:.1%}  Lineups: {sample.lineups_focus_ratio:.1%}')
print(f'  App opens/week: {sample.app_opens_per_week}')

# Club distribution
club_dist = Counter(p.favorite_club for p in profiles.values())
print('\nTop 10 clubs by user count:')
for club, cnt in club_dist.most_common(10):
    print(f'  {club}: {cnt}')

## 7. Merged player stats (Bayern) — top by goal participations

In [ ]:
merged = load_player_stats('DFL-CLU-00000G')
top = sorted(merged, key=lambda p: p.goal_participations, reverse=True)[:10]
print(f'{"Name":<25} {"Goals":>6} {"Assists":>8} {"G+A":>5} {"xG":>6} {"Pos":<12}')
print('-' * 68)
for p in top:
    print(f'{p.name:<25} {p.goals:>6} {p.assists:>8} {p.goal_participations:>5} {p.xg:>6.2f} {p.position:<12}')